# Bronze — Carga de JSON para Delta

Lê os arquivos JSON depositados pelo GitHub Actions no Volume,  
faz merge na tabela `judicial.bronze.datajud_raw` e move os arquivos para `processados/`.

**Pré-requisito:** `setup_environment` executado e arquivos presentes em `/Volumes/judicial/bronze/raw_files/{tribunal}/`

> Não usa DataFrame API (spark.read / withColumn). Todas as operações de dados são via `spark.sql()`.  
> Operações de arquivo (listar, mover) usam `os` e `shutil` via FUSE mount do Volume.

## 1. Configuração

In [0]:
import os
import shutil

VOLUME_BASE  = "/Volumes/judicial/bronze/raw_files"
BRONZE_TABLE = "judicial.bronze.datajud_raw"
TRIBUNAIS    = ["TJRS", "TJSC", "TJPR", "TJSP"]


## 2. Carga e merge por tribunal

In [0]:
for tribunal in TRIBUNAIS:
    source_dir    = f"{VOLUME_BASE}/{tribunal}"
    processed_dir = f"{VOLUME_BASE}/processados/{tribunal}"

    # ── Verificar arquivos disponíveis ────────────────────────────────────
    try:
        arquivos = [f for f in os.listdir(source_dir) if f.endswith(".json")]
    except FileNotFoundError:
        print(f"{tribunal}: diretório não encontrado, pulando")
        continue

    if not arquivos:
        print(f"{tribunal}: nenhum arquivo para processar")
        continue

    print(f"\n {tribunal} - {len(arquivos)} arquivo(s) encontrado(s)")

    # ── Criar tabela ou fazer merge via SQL puro ──────────────────────────
    tabela_existe = spark.catalog.tableExists(BRONZE_TABLE)

    if not tabela_existe:
        spark.sql(f"""
            CREATE TABLE {BRONZE_TABLE}
            USING DELTA
            AS
            SELECT
                *,
                '{tribunal}'        AS tribunal,
                current_timestamp() AS data_ingestao
            FROM read_files(
                '{source_dir}/',
                format        => 'json',
                multiline      => true,
                mergeSchema   => true
            )
        """)
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {BRONZE_TABLE}").collect()[0]["n"]
        print(f"  Tabela criada: {total:,} registros")

    else:
        spark.sql(f"""
            MERGE INTO {BRONZE_TABLE} AS t
            USING (
                SELECT
                    *,
                    '{tribunal}'        AS tribunal,
                    current_timestamp() AS data_ingestao
                FROM read_files(
                    '{source_dir}/',
                    format        => 'json',
                    multiline    => true,
                    mergeSchema   => true
                )
            ) AS s
            ON t._id = s._id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        total = spark.sql(f"""
            SELECT COUNT(*) AS n FROM {BRONZE_TABLE}
            WHERE tribunal = '{tribunal}'
        """).collect()[0]["n"]
        print(f"  Merge concluído: {total:,} registros em {tribunal}")

    # ── Mover arquivos para processados/ ─────────────────────────────────
    os.makedirs(processed_dir, exist_ok=True)
    for arquivo in arquivos:
        shutil.move(f"{source_dir}/{arquivo}", f"{processed_dir}/{arquivo}")
        print(f"  Movido: '{arquivo}' para 'processados/{tribunal}/'")


## 3. Verificação

In [0]:
spark.sql("""
    SELECT tribunal, COUNT(*) AS total, MAX(data_ingestao) AS ultima_ingestao
    FROM judicial.bronze.datajud_raw
    GROUP BY tribunal
""").show()
